
# FER2013 Emotion Recognition – CNN + Transfer Learning

This notebook builds an emotion recognition model using the **FER2013 dataset**.  
It intentionally creates a **very small training set**:

- **20 clean images per emotion**
- **10 noisy images per emotion (Gaussian noise augmentation)**
- **All remaining images → validation set**

This design simulates **low-data learning** and helps evaluate robustness.

The notebook includes:

- Dataset preprocessing
- Controlled dataset sampling
- Data augmentation
- Baseline CNN
- Transfer Learning (MobileNetV2)
- Early stopping & checkpoints
- Evaluation metrics
- Confusion matrix
- Precision / Recall / F1-score
- Model export


## 1. Install & Import Libraries

In [ ]:

import os
import zipfile
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


## 2. Extract Dataset

In [ ]:

zip_path = "archive.zip"   # path to Kaggle FER2013 zip
extract_path = "fer_dataset"

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

print("Dataset extracted.")


## 3. Prepare Custom Train / Validation Split

In [ ]:

base_dir = os.path.join(extract_path, "train")

train_dir = "custom_train"
val_dir = "custom_val"

classes = os.listdir(base_dir)

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

for c in classes:
    os.makedirs(os.path.join(train_dir, c), exist_ok=True)
    os.makedirs(os.path.join(val_dir, c), exist_ok=True)

    images = os.listdir(os.path.join(base_dir, c))
    random.shuffle(images)

    train_clean = images[:20]
    noisy_source = images[20:30]
    val_images = images[30:]

    # Copy clean images
    for img in train_clean:
        shutil.copy(os.path.join(base_dir, c, img),
                    os.path.join(train_dir, c, img))

    # Create noisy images
    import cv2
    for img in noisy_source:
        img_path = os.path.join(base_dir, c, img)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        noise = np.random.normal(0, 25, image.shape)
        noisy = image + noise
        noisy = np.clip(noisy, 0, 255).astype(np.uint8)

        save_path = os.path.join(train_dir, c, "noisy_" + img)
        cv2.imwrite(save_path, noisy)

    # Validation images
    for img in val_images:
        shutil.copy(os.path.join(base_dir, c, img),
                    os.path.join(val_dir, c, img))

print("Custom dataset created.")


## 4. Data Preprocessing & Augmentation

In [ ]:

IMG_SIZE = 48
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="rgb",
    class_mode="categorical",
    shuffle=False
)


## 5. Baseline CNN Model

In [ ]:

baseline_model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(48,48,3)),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(64,(3,3),activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    Conv2D(128,(3,3),activation='relu'),
    BatchNormalization(),
    MaxPooling2D(),

    Flatten(),
    Dense(256,activation='relu'),
    Dropout(0.5),
    Dense(7,activation='softmax')
])

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

baseline_model.summary()


## 6. Training (Baseline CNN)

In [ ]:

early_stop = EarlyStopping(patience=5, restore_best_weights=True)

checkpoint = ModelCheckpoint(
    "best_baseline_model.h5",
    monitor="val_accuracy",
    save_best_only=True
)

history = baseline_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stop, checkpoint]
)


## 7. Training Graphs

In [ ]:

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Accuracy")
plt.legend(["Train","Validation"])
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Loss")
plt.legend(["Train","Validation"])
plt.show()


## 8. Transfer Learning – MobileNetV2

In [ ]:

base_model = MobileNetV2(
    input_shape=(48,48,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
outputs = Dense(7, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 9. Train Transfer Learning Model

In [ ]:

checkpoint = ModelCheckpoint(
    "best_transfer_model.h5",
    monitor="val_accuracy",
    save_best_only=True
)

history_tl = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stop, checkpoint]
)


## 10. Model Evaluation

In [ ]:

val_generator.reset()
preds = model.predict(val_generator)
y_pred = np.argmax(preds, axis=1)
y_true = val_generator.classes

print(classification_report(y_true, y_pred, target_names=list(train_generator.class_indices.keys())))


## 11. Confusion Matrix

In [ ]:

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=train_generator.class_indices.keys(),
            yticklabels=train_generator.class_indices.keys())
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


## 12. Save Model

In [ ]:

model.save("emotion_model_final.h5")
print("Model saved.")



# Future Improvements for Classroom Engagement Detection

To adapt this model for **student engagement monitoring**:

1. Combine emotion detection with **head pose detection**
2. Add **eye gaze tracking**
3. Use **temporal models (LSTM/Transformer)** for behavior over time
4. Integrate **attention estimation**
5. Use **classroom video streams** instead of static images
